# 7 Interrupts (p109)
# 7.1 Introduction
ARM has two types of interrupt sources:
1. Interrupts coming from GPU peripherals.
2. Interrupts coming from local ARM control peripherals.
The ARM processor gets three types of interrupts:
1. Interrupts from ARM specific peripherals.
2. Interrupts from GPU peripherals.
3. Special events interrupts.
The ARM specific interrupts are:
- One timer.
- One Mailbox.
- Two Doorbells.
- Two GPU halted interrupts.
- Two Address/access error interrupt
The Mailbox and Doorbell registers are not for general usage.
For each interrupt source (ARM or GPU) there is an interrupt enable bit (read/write) and an interrupt pending bit (Read Only). All interrupts generated by the arm control block are level sensitive interrupts. Thus all interrupts remain asserted until disabled or the interrupt source is cleared.
Default the interrupts from doorbell 0,1 and mailbox 0 go to the ARM this means that these resources should be written by the GPU and read by the ARM. The opposite holds for doorbells 2, 3 and mailbox 1.

# 7.2 Interrupt pending   
- todo  
# 7.3 Fast interrupt (FIQ)
The ARM also supports a Fast Interrupt (FIQ). One interrupt sources can be selected to be connected to the ARM FIQ input. There is also one FIQ enable. An interrupt which is selected as FIQ should have its normal interrupt enable bit cleared. Otherwise an normal and a FIQ interrupt will be fired at the
same time. Not a good idea!

# 7.4 interrupt priority  (p110)
There is no priority for any interrupt. If one interrupt is much more important then all others it can be routed to the **FIQ**. Any remaining interrupts have to be processed by **polling the pending** registers. It is up to the ARM software to devise a strategy. e.g. First start looking for specific **pending bits** or process them **all shifting one bit at a time**.

As interrupt may arrive whilst this process is ongoing the usual care for any 'race-condition critical' code must be taken. The following ARM assembly code has been proven to work:

- tododododo 

# 7.5 register (p112)  (peripherals/irq.h/arm_irq_regs)
- [yt irq](https://youtu.be/nUW1FB_5vqo?list=PLVxiWMqQvhg9FCteL7I0aohj1_YiUx1x8&t=821)
- The base address for the ARM interrupt register is 0x7E00B000
  - 所以 irq.h 的 offset 是 B000 + 200 = B200
- register overview (irq.h/arm_irq_regs_2837) 

|Address offset|Name              |
|--------------|------------------|
|0x200         |IRQ basic pending |
|0x204         |IRQ pending 1     |
|0x208         |IRQ pending 2     |
|0x20C         |FIQ control       |
|0x210         |Enable IRQs 1     |
|0x214         |Enable IRQs 2     |
|0x218         |Enable basic IRQs |
|0x21C         |Disable IRQs 1    |
|0x220         |Disable IRQs 2    |
|0x224         |Disable Basic IRQs|

The following is a table which lists all interrupts which can come from the peripherals which can be handled by the ARM.

ARM peripherals interrupts table    

|# |IRQ 0-15        |# |IRQ 16-31  |# |IRQ 32-47      |# |IRQ 48-63      |
|--|----------------|--|-----------|--|---------------|--|---------------|
|0 |Timer 0         |16|DMA 0      |32|HDMI CEC       |48|**smi**        |
|1 |Timer 1         |17|DMA 1      |33|HVS            |49|**gpio_int[0]**|
|2 |Timer 2         |18|DMA 2      |34|RPIVID         |50|**gpio_int[1]**|
|3 |Timer 3         |19|DMA 3      |35|SDC            |51|**gpio_int[2]**|
|4 |H264  0         |20|DMA 4      |36|DSI 0          |52|**gpio_int[3]**|
|5 |H264  1         |21|DMA 5      |37|Pixel Valve 2  |53|**i2c_int**    |
|6 |H264  2         |22|DMA 6      |38|Camera 0       |54|**spi_int**    |
|7 |JPEG            |23|DMA 7 & 8  |39|Camera 1       |55|**pcm_int**    |
|8 |ISP             |24|DMA 9 & 10 |40|HDMI 0         |56|SDHOST         |
|9 |USB             |25|DMA 11     |41|HDMI 1         |57|**uart_int**   |
|10|V3D             |26|DMA 12     |42|Pixel Valve 3  |58|ETH_PCIe L2    |
|11|Transposer      |27|DMA 13     |43|**i2c_spi_slv**|59|VEC            |
|12|Multicore Sync 0|28|DMA 14     |44|DSI 1          |60|CPG            |
|13|Multicore Sync 1|29|**AUX int**|45|**pwa0**       |61|RNG            |
|14|Multicore Sync 2|30|ARM        |46|**pwa1**       |62|EMMC & EMMC2   |
|15|Multicore Sync 3|31|DMA 15     |47|CPR            |63|ETH_PCIe secure|

- The table above has many empty entries. These should not be enabled as they will interfere with the GPU operation.  ARM peripherals interrupts table.

- 雖然這樣說  但是我加上一些 不是 boldface 的 interrupt 是在舊版的說明說 bcm2711 (p86  6.2.4 videoCore interrupts) 找到的, 其中 0: Timer0, 1:Timer1 在 irq.h: vc_irqs 中都有被用到